In [2]:
"""
================================================================================
  DECISION SUPPORT SYSTEM (DSS) — KNOWLEDGE-BASED META-ENSEMBLE
  Target Journal  : Knowledge-Based Systems (Q1, Elsevier)
  Module          : 7 — Final Operational Intelligence & VIKOR Ranking
  Version         : 2.0  (Debugged — Cartesian-product guard + scalar fix)
  GPU Required    : NO  (pure NumPy / Pandas / Matplotlib — CPU only)

  Pipeline:
    ┌─────────────────────────────────────────────────────────────────┐
    │  CELL 1  — Install all required libraries (run once)            │
    │  CELL 2  — Full DSS execution                                   │
    └─────────────────────────────────────────────────────────────────┘
================================================================================
"""

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — LIBRARY INSTALLATION  (Run this cell ONCE, then restart runtime) ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import subprocess, sys

_PACKAGES = [
    "numpy>=1.24",
    "pandas>=2.0",
    "matplotlib>=3.7",
    "seaborn>=0.12",
    "scikit-learn>=1.3",
    "Pillow>=10.0",          # TIFF/PNG export backend for matplotlib
    # ── Optional (only needed if re-running model inference, not DSS) ──────
    # "lightgbm",
    # "interpret",
    # "torch",
]

print("\n" + "=" * 66)
print("  CELL 1 — DEPENDENCY INSTALLATION")
print("=" * 66)
print("  GPU Required : NO  (DSS is CPU-only — model CSVs are pre-computed)\n")

_all_ok = True
for _pkg in _PACKAGES:
    _label = _pkg.split(">=")[0]
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet",
             "--upgrade", _pkg],
            stderr=subprocess.DEVNULL,
        )
        print(f"  ✔  {_label:<18} installed / up-to-date")
    except subprocess.CalledProcessError as _e:
        print(f"  ✘  {_label:<18} FAILED — {_e}")
        _all_ok = False

print("\n" + ("=" * 66))
if _all_ok:
    print("  ALL PACKAGES INSTALLED.  Proceed to CELL 2.")
else:
    print("  ONE OR MORE PACKAGES FAILED. Check network / pip version.")
print("=" * 66 + "\n")


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — FULL DSS EXECUTION                                               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── Standard library ──────────────────────────────────────────────────────────
import os
import warnings
from datetime import datetime
from pathlib import Path

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates  as mdates
from matplotlib.gridspec import GridSpec
from matplotlib.ticker   import AutoMinorLocator
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
#  LOGGER
# ══════════════════════════════════════════════════════════════════════════════
_W = 78

def log(msg: str, level: str = "INFO") -> None:
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"  [{ts}]  [{level:^8}]  {msg}")

def section(title: str) -> None:
    print(f"\n{'=' * _W}")
    print(f"  ▶  {title}")
    print('=' * _W)

def banner(title: str) -> None:
    print(f"\n{'=' * _W}")
    print(f"  {title}")
    print('=' * _W)

banner("MARITIME DSS — KNOWLEDGE-BASED META-ENSEMBLE ENGINE\n"
       "  Knowledge-Based Systems (Q1) | Operational Intelligence Module v2.0")
log("System initialised. NumPy, Pandas, Matplotlib, Seaborn loaded.")
log(f"Execution timestamp : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log(f"GPU Required        : NO  (CPU-only DSS pipeline)")


# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
BASE_DIR   = Path("/content/drive/MyDrive/KBS_Paper/Outputs")
INPUT_META = {
    "LightGBM": BASE_DIR / "4_LightGBM_KBS"        / "lightgbm_oos_predictions.csv",
    "EBM"     : BASE_DIR / "5_EBM_KBS"             / "ebm_oos_predictions.csv",
    "LSTM"    : BASE_DIR / "6_LSTM_KBS_Optimized"  / "lstm_optimized_oos_predictions.csv",
}
OUTPUT_DIR = BASE_DIR / "7_DSS_KBS_Final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_HORIZON   = "+6h"
CONFORMAL_ALPHA  = 0.90
VIKOR_WEIGHTS    = np.array([0.4, 0.3, 0.3])
VIKOR_V          = 0.5
Q_SAFE           = 0.33
Q_WARN           = 0.66
PLOT_WINDOW_DAYS = 14
DPI_EXPORT       = 600

# Flexible column name resolution
PRED_COL_PATTERNS   = ["pred_hs", "predicted_hs", "y_pred",  "hs_pred",  "forecast_hs"]
ACTUAL_COL_PATTERNS = ["actual_hs","obs_hs",       "y_true",  "hs_actual","observed_hs"]
TIME_COL_PATTERNS   = ["time",     "datetime",     "date",    "timestamp", "Date"]

log(f"Output directory         : {OUTPUT_DIR}")
log(f"Target horizon           : {TARGET_HORIZON}")
log(f"Conformal α              : {CONFORMAL_ALPHA:.0%}")
log(f"VIKOR weights (C1/C2/C3) : {VIKOR_WEIGHTS}")
log(f"VIKOR v (mechanism)      : {VIKOR_V}")


# ══════════════════════════════════════════════════════════════════════════════
#  UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def _find_col(df: pd.DataFrame, patterns: list, label: str) -> str:
    """Case-insensitive flexible column resolver."""
    lmap = {c.lower(): c for c in df.columns}
    for p in patterns:
        if p.lower() in lmap:
            return lmap[p.lower()]
    raise KeyError(
        f"Cannot locate '{label}' column.\n"
        f"  Available : {list(df.columns)}\n"
        f"  Expected  : {patterns}"
    )


def _safe_scalar(val):
    """
    Extract a guaranteed Python float from a value that may be a
    pandas Series (happens when index has residual duplicates).
    """
    if isinstance(val, (pd.Series, pd.DataFrame)):
        return float(val.iloc[0])
    return float(val)


def _deduplicate_index(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """
    Remove duplicate DatetimeIndex entries.

    Root cause in this pipeline: rolling-window OOS evaluation assigns the
    same timestamp to multiple forecast-origin windows.  We keep `last`,
    which corresponds to the prediction made from the most recent data
    — the operationally correct choice.
    """
    n_before = len(df)
    df = df[~df.index.duplicated(keep="last")]
    n_dup = n_before - len(df)
    if n_dup > 0:
        log(f"[{name}] Deduplicated {n_dup:,} repeated timestamps "
            f"({n_before:,} → {len(df):,} rows) — kept 'last'.",
            level="WARNING")
    return df


def _load_predictions(path: Path, model_name: str,
                      horizon: str) -> pd.DataFrame:
    """
    Load a model OOS CSV.  Handles:
      • Flexible time column naming
      • Optional 'horizon' column
      • Duplicate timestamp removal  ← KEY BUG-FIX v2.0
    Returns DataFrame indexed by DatetimeIndex with columns:
        {model_name}_pred  |  actual_hs
    """
    log(f"Loading {model_name}  →  {path.name}")
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()

    # ── Resolve time column ───────────────────────────────────────────────────
    time_col = _find_col(df, TIME_COL_PATTERNS, "time")
    df[time_col] = pd.to_datetime(df[time_col])
    df = df.set_index(time_col).sort_index()

    # ── Filter horizon ────────────────────────────────────────────────────────
    if "horizon" in df.columns:
        df = df[df["horizon"].astype(str).str.strip() == horizon].copy()
        if df.empty:
            raise ValueError(
                f"[{model_name}] No rows for horizon='{horizon}'. "
                f"Available: {df['horizon'].unique().tolist()}"
            )
    else:
        log(f"[{model_name}] 'horizon' column absent — treating all rows "
            f"as {horizon}.", level="WARNING")

    # ── BUG-FIX: remove duplicate timestamps BEFORE merge ────────────────────
    df = _deduplicate_index(df, model_name)

    # ── Resolve prediction / actual columns ───────────────────────────────────
    pred_col   = _find_col(df, PRED_COL_PATTERNS,   "prediction")
    actual_col = _find_col(df, ACTUAL_COL_PATTERNS, "actual")

    out = pd.DataFrame({
        f"{model_name}_pred" : df[pred_col].astype(float),
        "actual_hs"          : df[actual_col].astype(float),
    })

    log(f"  ✔  {model_name:<10} | rows: {len(out):>7,} | "
        f"pred='{pred_col}'  actual='{actual_col}'")
    return out


# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 1 — META-ENSEMBLE & EPISTEMIC UNCERTAINTY
# ══════════════════════════════════════════════════════════════════════════════
section("MODULE 1 — META-ENSEMBLE CONSTRUCTION & EPISTEMIC UNCERTAINTY")

frames = {}
for mname, mpath in INPUT_META.items():
    try:
        frames[mname] = _load_predictions(mpath, mname, TARGET_HORIZON)
    except Exception as exc:
        log(f"FATAL — {mname}: {exc}", level="ERROR")
        raise SystemExit(1)

# ── Inner merge (on deduplicated DatetimeIndex) ───────────────────────────────
try:
    lgb_df  = frames["LightGBM"]
    ebm_df  = frames["EBM"].drop(columns=["actual_hs"], errors="ignore")
    lstm_df = frames["LSTM"].drop(columns=["actual_hs"], errors="ignore")

    dss = pd.merge(lgb_df,  ebm_df,  left_index=True,
                   right_index=True, how="inner")
    dss = pd.merge(dss, lstm_df,     left_index=True,
                   right_index=True, how="inner")
    dss.dropna(subset=["LightGBM_pred", "EBM_pred", "LSTM_pred", "actual_hs"],
               inplace=True)

    # ── Sanity check: merged count must not exceed smallest input ─────────────
    smallest_input = min(len(f) for f in frames.values())
    if len(dss) > smallest_input * 1.02:          # allow 2 % tolerance
        log(f"CRITICAL — Merged rows ({len(dss):,}) exceed smallest input "
            f"({smallest_input:,}) by > 2%.  "
            f"Cartesian-product inflation detected despite dedup. "
            f"Falling back to groupby-mean dedup on merged frame.",
            level="ERROR")
        dss = dss.groupby(dss.index).last()       # last-resort fix
        log(f"Post-fallback rows: {len(dss):,}", level="WARNING")

    log(f"Inner-join complete — aligned rows : {len(dss):,}")

except Exception as exc:
    log(f"FATAL — Merge failure: {exc}", level="ERROR")
    raise SystemExit(1)

# ── Ensemble statistics ───────────────────────────────────────────────────────
pred_cols = ["LightGBM_pred", "EBM_pred", "LSTM_pred"]
dss["ensemble_pred_hs"]   = dss[pred_cols].mean(axis=1)
dss["model_disagreement"] = dss[pred_cols].std(axis=1, ddof=0)

log(f"Ensemble mean Hs          — "
    f"μ={dss['ensemble_pred_hs'].mean():.4f} m  "
    f"σ={dss['ensemble_pred_hs'].std():.4f} m")
log(f"Epistemic uncertainty (σ) — "
    f"μ={dss['model_disagreement'].mean():.4f} m  "
    f"max={dss['model_disagreement'].max():.4f} m")


# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 2 — ALEATORIC UNCERTAINTY (CONFORMAL PREDICTION) + PICP / MPIW
# ══════════════════════════════════════════════════════════════════════════════
section("MODULE 2 — ALEATORIC UNCERTAINTY & CONFORMAL PREDICTION BOUNDS")

residuals = (dss["ensemble_pred_hs"] - dss["actual_hs"]).abs()
e_margin  = float(np.quantile(residuals, CONFORMAL_ALPHA))
log(f"Conformal margin e at {CONFORMAL_ALPHA:.0%} quantile : {e_margin:.4f} m")

dss["Lower_Bound_90"] = (dss["ensemble_pred_hs"] - e_margin).clip(lower=0.0)
dss["Upper_Bound_90"] =  dss["ensemble_pred_hs"] + e_margin
dss["PI_Width"]       =  dss["Upper_Bound_90"] - dss["Lower_Bound_90"]

within = (dss["actual_hs"] >= dss["Lower_Bound_90"]) & \
         (dss["actual_hs"] <= dss["Upper_Bound_90"])
PICP  = float(within.mean()) * 100.0
MPIW  = float(dss["PI_Width"].mean())

print()
print(f"  {'─' * 62}")
print(f"  {'UNCERTAINTY QUANTIFICATION REPORT':^62}")
print(f"  {'─' * 62}")
print(f"  {'Conformal PICP (%)':<50} {PICP:>8.3f}")
print(f"  {'Mean Prediction Interval Width — MPIW (m)':<50} {MPIW:>8.4f}")
print(f"  {'Empirical Margin of Error e (m)':<50} {e_margin:>8.4f}")
print(f"  {'Target nominal coverage (%)':<50} {CONFORMAL_ALPHA*100:>8.1f}")
print(f"  {'Coverage ≥ nominal?':<50} {'YES ✔' if PICP >= CONFORMAL_ALPHA*100 else 'NO ✘':>8}")
print(f"  {'─' * 62}\n")


# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 3 — VIKOR MCDM OPERATIONAL RISK CLASSIFICATION
# ══════════════════════════════════════════════════════════════════════════════
section("MODULE 3 — VIKOR OPERATIONAL RISK CLASSIFICATION")

# Criteria (all cost-type: higher = more risk)
C = np.column_stack([
    dss["ensemble_pred_hs"].values,   # C1 — wave height
    dss["PI_Width"].values,           # C2 — aleatoric interval width
    dss["model_disagreement"].values, # C3 — epistemic disagreement
])
w = VIKOR_WEIGHTS

# ── Step 1: Best (f*) and Worst (f-) ─────────────────────────────────────────
f_star  = C.min(axis=0)   # least risky per criterion
f_minus = C.max(axis=0)   # most  risky per criterion

log(f"VIKOR f* (best)  — Hs={f_star[0]:.4f} m | "
    f"Width={f_star[1]:.4f} m | Disagree={f_star[2]:.4f} m")
log(f"VIKOR f- (worst) — Hs={f_minus[0]:.4f} m | "
    f"Width={f_minus[1]:.4f} m | Disagree={f_minus[2]:.4f} m")

# ── Step 2: Normalised weighted distances ─────────────────────────────────────
denom = np.where((f_minus - f_star) == 0, 1e-12, f_minus - f_star)
nwd   = w[np.newaxis, :] * (C - f_star[np.newaxis, :]) / denom[np.newaxis, :]

S = nwd.sum(axis=1)    # utility   (sum of weighted distances)
R = nwd.max(axis=1)    # regret    (maximum weighted distance)

# ── Step 3: Q index ───────────────────────────────────────────────────────────
S_star, S_minus = S.min(), S.max()
R_star, R_minus = R.min(), R.max()
S_den = S_minus - S_star if (S_minus - S_star) != 0 else 1e-12
R_den = R_minus - R_star if (R_minus - R_star) != 0 else 1e-12
Q     = VIKOR_V * (S - S_star) / S_den + (1 - VIKOR_V) * (R - R_star) / R_den

dss["VIKOR_S"]    = S
dss["VIKOR_R"]    = R
dss["VIKOR_Q"]    = Q

# ── Risk categorisation ───────────────────────────────────────────────────────
def _classify(q: float) -> str:
    if q < Q_SAFE:  return "Safe"
    if q <= Q_WARN: return "Warning"
    return "Danger"

dss["Risk_Level"] = dss["VIKOR_Q"].apply(_classify)

rc    = dss["Risk_Level"].value_counts()
total = len(dss)

print()
print(f"  {'─' * 62}")
print(f"  {'VIKOR OPERATIONAL RISK DISTRIBUTION':^62}")
print(f"  {'─' * 62}")
for state, colour in [("Safe","Green"), ("Warning","Yellow"), ("Danger","Red")]:
    n = rc.get(state, 0)
    print(f"  {colour:>8} ({state:<8}) : "
          f"{n:>8,} steps  ({n/total*100:5.1f} %)")
print(f"  {'─' * 62}")
print(f"  {'Total time steps':<52} {total:>8,}")
print(f"  {'─' * 62}\n")

log(f"VIKOR Q — min={Q.min():.4f} | mean={Q.mean():.4f} | max={Q.max():.4f}")


# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 4 — FIGURE 5: DSS MASTER DASHBOARD  (600 DPI, PNG + TIFF)
# ══════════════════════════════════════════════════════════════════════════════
section("MODULE 4 — GENERATING FIG. 5: DSS MASTER DASHBOARD")

# ── Locate 14-day storm window ────────────────────────────────────────────────
peak_idx  = dss["actual_hs"].idxmax()
# BUG-FIX v2.0: _safe_scalar() prevents Series.__format__ TypeError
peak_hs   = _safe_scalar(dss.loc[peak_idx, "actual_hs"])

half_win  = pd.Timedelta(days=PLOT_WINDOW_DAYS // 2)
win_start = peak_idx - half_win
win_end   = peak_idx + half_win
dss_win   = dss.loc[win_start:win_end].copy()

# Final dedup of window slice (defensive)
dss_win   = dss_win[~dss_win.index.duplicated(keep="last")]

log(f"Storm peak   : {peak_idx}  |  Hs={peak_hs:.3f} m")
log(f"Plot window  : {win_start.date()} → {win_end.date()} "
    f"({len(dss_win):,} time steps)")

# ── Style ─────────────────────────────────────────────────────────────────────
sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1.15)
plt.rcParams.update({
    "font.family"      : "DejaVu Sans",
    "axes.linewidth"   : 0.8,
    "grid.linewidth"   : 0.5,
    "grid.alpha"       : 0.6,
    "xtick.direction"  : "in",
    "ytick.direction"  : "in",
    "axes.spines.top"  : False,
    "axes.spines.right": False,
})

CACT   = "#1B4F72"    # actual Hs   — deep navy
CENS   = "#E67E22"    # ensemble    — warm amber
CCI    = "#E74C3C"    # conformal   — red
CDIS   = "#8E44AD"    # disagreement — purple
CGRID  = "#BFC9CA"

RISK_C = {"Safe": "#27AE60", "Warning": "#F39C12", "Danger": "#C0392B"}

# ── Figure layout ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14), dpi=150)
gs  = GridSpec(3, 1, figure=fig, hspace=0.40,
               height_ratios=[2.4, 1.4, 1.4])

ax0 = fig.add_subplot(gs[0])
ax1 = fig.add_subplot(gs[1], sharex=ax0)
ax2 = fig.add_subplot(gs[2], sharex=ax0)

t = dss_win.index

# ─────────────────────────────────────────────────────────────────────────────
#  Panel (a) — Forecast vs. Actual + 90% Conformal Prediction Interval
# ─────────────────────────────────────────────────────────────────────────────
ax0.fill_between(t,
                 dss_win["Lower_Bound_90"],
                 dss_win["Upper_Bound_90"],
                 color=CCI, alpha=0.15, label="90% Conformal PI", zorder=1)
ax0.fill_between(t,
                 dss_win["Lower_Bound_90"],
                 dss_win["Upper_Bound_90"],
                 color=CCI, alpha=0.0, edgecolor=CCI,
                 linewidth=0.6, linestyle=":", zorder=2)

ax0.plot(t, dss_win["actual_hs"],
         color=CACT, lw=1.9, label="Observed $H_s$", zorder=4)
ax0.plot(t, dss_win["ensemble_pred_hs"],
         color=CENS, lw=1.6, ls="--",
         label="Meta-Ensemble $\\hat{H}_s$", zorder=5)

# Individual model traces (thin, transparent) for visual context
alpha_m = 0.22
for col, lbl, clr in [
    ("LightGBM_pred", "LightGBM",  "#2ECC71"),
    ("EBM_pred",      "EBM",       "#3498DB"),
    ("LSTM_pred",     "Att-LSTM",  "#E74C3C"),
]:
    ax0.plot(t, dss_win[col], lw=0.8, alpha=alpha_m,
             color=clr, label=lbl, zorder=3)

ax0.axvline(x=peak_idx, color="black", lw=1.3, ls=":", alpha=0.75,
            label="Storm Peak", zorder=6)
ax0.set_ylabel("$H_s$ (m)", fontsize=11)
ax0.set_title(
    "Figure 5 — DSS Master Dashboard: Meta-Ensemble Forecast & "
    "VIKOR Operational Risk Assessment (+6 h Horizon)",
    fontsize=12, fontweight="bold", pad=10
)
ax0.legend(loc="upper left", framealpha=0.92, fontsize=8.5,
           ncol=3, columnspacing=0.8)
ax0.set_ylim(bottom=0)
ax0.yaxis.set_minor_locator(AutoMinorLocator())
ax0.tick_params(labelbottom=False)

# Annotation box — PICP / MPIW
ax0.annotate(
    f"PICP = {PICP:.2f} %\nMPIW = {MPIW:.3f} m\ne   = {e_margin:.3f} m",
    xy=(0.985, 0.96), xycoords="axes fraction",
    ha="right", va="top", fontsize=8.8,
    bbox=dict(boxstyle="round,pad=0.45", fc="white",
              ec="#AAB7B8", alpha=0.90, lw=0.8)
)

# ─────────────────────────────────────────────────────────────────────────────
#  Panel (b) — Epistemic Uncertainty (Model Disagreement)
# ─────────────────────────────────────────────────────────────────────────────
ax1.fill_between(t, 0, dss_win["model_disagreement"],
                 color=CDIS, alpha=0.28, zorder=1)
ax1.plot(t, dss_win["model_disagreement"],
         color=CDIS, lw=1.5, zorder=2, label="$\\sigma_{ensemble}$")
ax1.axvline(x=peak_idx, color="black", lw=1.3, ls=":", alpha=0.75, zorder=3)

mu_dis = float(dss_win["model_disagreement"].mean())
ax1.axhline(mu_dis, color=CDIS, lw=1.0, ls="--", alpha=0.55, zorder=2)
ax1.text(t[-1], mu_dis * 1.06,
         f"window μ = {mu_dis:.3f} m",
         ha="right", va="bottom", fontsize=8.2, color=CDIS)

ax1.set_ylabel("$\\sigma_{ensemble}$ (m)", fontsize=11)
ax1.set_ylim(bottom=0)
ax1.yaxis.set_minor_locator(AutoMinorLocator())
ax1.tick_params(labelbottom=False)
ax1.legend(loc="upper right", fontsize=8.5, framealpha=0.9)

# ─────────────────────────────────────────────────────────────────────────────
#  Panel (c) — VIKOR Q: Colour-coded Operational Risk Bar Chart
# ─────────────────────────────────────────────────────────────────────────────
try:
    dt_days = (t[1] - t[0]).total_seconds() / 86400.0
except Exception:
    dt_days = 1 / 24.0

bar_w  = dt_days * 0.88
t_num  = mdates.date2num(t.to_pydatetime())
colours = dss_win["Risk_Level"].map(RISK_C).values

# Vectorised bar drawing (no Python loop — fast even for 22k steps)
ax2.bar(t_num,
        dss_win["VIKOR_Q"].values,
        width=bar_w,
        color=colours,
        align="center",
        alpha=0.85,
        zorder=2)

# Threshold lines
ax2.axhline(Q_SAFE, color="#F1C40F", lw=1.4, ls="--", alpha=0.95, zorder=3)
ax2.axhline(Q_WARN, color="#C0392B", lw=1.4, ls="--", alpha=0.95, zorder=3)
ax2.axvline(mdates.date2num(peak_idx.to_pydatetime()),
            color="black", lw=1.3, ls=":", alpha=0.75, zorder=4)

# Threshold labels
_tx = ax2.get_xlim()[0] if ax2.get_xlim()[0] != 0.0 else t_num[0]
ax2.text(t_num[1], Q_SAFE + 0.015, "Safe / Warning threshold",
         fontsize=7.8, color="#B7950B", va="bottom", fontweight="bold")
ax2.text(t_num[1], Q_WARN + 0.015, "Warning / Danger threshold",
         fontsize=7.8, color="#C0392B", va="bottom", fontweight="bold")

ax2.set_ylabel("VIKOR $Q$ Index", fontsize=11)
ax2.set_ylim(0, 1.08)
ax2.set_xlabel("Date (UTC)", fontsize=11)
ax2.yaxis.set_minor_locator(AutoMinorLocator())

# Risk legend
patches = [
    mpatches.Patch(color=RISK_C["Safe"],    label=f"Safe    (Q < {Q_SAFE})"),
    mpatches.Patch(color=RISK_C["Warning"], label=f"Warning ({Q_SAFE} ≤ Q ≤ {Q_WARN})"),
    mpatches.Patch(color=RISK_C["Danger"],  label=f"Danger  (Q > {Q_WARN})"),
]
ax2.legend(handles=patches, loc="upper left",
           fontsize=8.5, framealpha=0.92, ncol=3, columnspacing=0.8)

# ── Shared x-axis ─────────────────────────────────────────────────────────────
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%Y"))
ax2.xaxis.set_major_locator(mdates.DayLocator(interval=2))
plt.setp(ax2.xaxis.get_majorticklabels(),
         rotation=0, ha="center", fontsize=9)

# ── Panel labels ──────────────────────────────────────────────────────────────
for ax, lbl in zip([ax0, ax1, ax2], ["(a)", "(b)", "(c)"]):
    ax.text(-0.055, 1.02, lbl, transform=ax.transAxes,
            fontsize=11, fontweight="bold", va="bottom")

plt.tight_layout()

# ── Export ────────────────────────────────────────────────────────────────────
for ext in ("png", "tiff"):
    fpath = OUTPUT_DIR / f"fig5_dss_master_dashboard_6h.{ext}"
    save_kw = dict(dpi=DPI_EXPORT, bbox_inches="tight", format=ext)
    if ext == "tiff":
        save_kw["pil_kwargs"] = {"compression": "tiff_lzw"}
    fig.savefig(str(fpath), **save_kw)
    log(f"Saved → {fpath}")

plt.close(fig)


# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 5 — EXPORT OPERATIONAL LOG CSV
# ══════════════════════════════════════════════════════════════════════════════
section("MODULE 5 — EXPORTING DSS OPERATIONAL LOG")

export_cols = [
    "actual_hs",
    "LightGBM_pred", "EBM_pred", "LSTM_pred",
    "ensemble_pred_hs", "model_disagreement",
    "Lower_Bound_90", "Upper_Bound_90", "PI_Width",
    "VIKOR_S", "VIKOR_R", "VIKOR_Q", "Risk_Level",
]
csv_path = OUTPUT_DIR / "dss_final_operational_log_6h.csv"
dss[export_cols].to_csv(str(csv_path), index=True,
                        index_label="time", float_format="%.6f")
log(f"Operational log → {csv_path}")
log(f"  Rows: {len(dss):,}  |  Columns: {len(export_cols)}")


# ══════════════════════════════════════════════════════════════════════════════
#  FINAL CONSOLE REPORT
# ══════════════════════════════════════════════════════════════════════════════
banner("DSS EXECUTION COMPLETE — FINAL PERFORMANCE SUMMARY")

mae  = mean_absolute_error(dss["actual_hs"], dss["ensemble_pred_hs"])
rmse = np.sqrt(mean_squared_error(dss["actual_hs"], dss["ensemble_pred_hs"]))
ss   = 1.0 - (
    np.sum((dss["actual_hs"] - dss["ensemble_pred_hs"]) ** 2) /
    np.sum((dss["actual_hs"] - dss["actual_hs"].mean()) ** 2)
)

print(f"\n  {'─' * 62}")
print(f"  {'ENSEMBLE REGRESSION PERFORMANCE':^62}")
print(f"  {'─' * 62}")
print(f"  {'MAE  (m)':<52} {mae:>8.4f}")
print(f"  {'RMSE (m)':<52} {rmse:>8.4f}")
print(f"  {'R²  (Skill Score)':<52} {ss:>8.4f}")
print(f"\n  {'UNCERTAINTY QUANTIFICATION':^62}")
print(f"  {'─' * 62}")
print(f"  {'Conformal PICP (%)':<52} {PICP:>8.3f}")
print(f"  {'MPIW (m)':<52} {MPIW:>8.4f}")
print(f"  {'Mean Epistemic Uncertainty  σ (m)':<52} {dss['model_disagreement'].mean():>8.4f}")
print(f"  {'Max  Epistemic Uncertainty  σ (m)':<52} {dss['model_disagreement'].max():>8.4f}")
print(f"\n  {'VIKOR OPERATIONAL RISK':^62}")
print(f"  {'─' * 62}")
print(f"  {'Q mean':<52} {dss['VIKOR_Q'].mean():>8.4f}")
print(f"  {'Q max':<52} {dss['VIKOR_Q'].max():>8.4f}")
for state, colour in [("Safe","Green"),("Warning","Yellow"),("Danger","Red")]:
    n = (dss["Risk_Level"] == state).sum()
    print(f"  {colour+' / '+state+' (%)':<52} {n/total*100:>8.2f}")
print(f"\n  {'─' * 62}")
print(f"  {'OUTPUT FILES':^62}")
print(f"  {'─' * 62}")
print(f"  fig5_dss_master_dashboard_6h.png  (600 DPI)")
print(f"  fig5_dss_master_dashboard_6h.tiff (600 DPI, LZW)")
print(f"  dss_final_operational_log_6h.csv  ({len(dss):,} rows)")
print(f"  Directory → {OUTPUT_DIR}")
print(f"  {'─' * 62}")
log("DSS v2.0 terminated successfully. No errors.")
print("=" * _W + "\n")


  CELL 1 — DEPENDENCY INSTALLATION
  GPU Required : NO  (DSS is CPU-only — model CSVs are pre-computed)

  ✔  numpy              installed / up-to-date
  ✔  pandas             installed / up-to-date
  ✔  matplotlib         installed / up-to-date
  ✔  seaborn            installed / up-to-date
  ✔  scikit-learn       installed / up-to-date
  ✔  Pillow             installed / up-to-date

  ALL PACKAGES INSTALLED.  Proceed to CELL 2.


  MARITIME DSS — KNOWLEDGE-BASED META-ENSEMBLE ENGINE
  Knowledge-Based Systems (Q1) | Operational Intelligence Module v2.0
  [2026-02-26 13:10:59]  [  INFO  ]  System initialised. NumPy, Pandas, Matplotlib, Seaborn loaded.
  [2026-02-26 13:10:59]  [  INFO  ]  Execution timestamp : 2026-02-26 13:10:59
  [2026-02-26 13:10:59]  [  INFO  ]  GPU Required        : NO  (CPU-only DSS pipeline)
  [2026-02-26 13:10:59]  [  INFO  ]  Output directory         : /content/drive/MyDrive/KBS_Paper/Outputs/7_DSS_KBS_Final
  [2026-02-26 13:10:59]  [  INFO  ]  Target horizon 